# Training DreamBooth and LoRA Models for Stable Diffusion

## Training DreamBooth and LoRA Models for Stable Diffusion

Run the following scripts:
```bash
uv run accelerate config default
```

Then execute the script to train DreamBooth model:
```bash
uv run accelerate launch train_dreambooth_lora_sdxl.py --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" --instance_data_dir="./datasets/try" --output_dir="./models/lora_xl_model" --mixed_precision="fp16" --instance_prompt="a photo of sks person" --resolution=1024 --train_batch_size=2 --gradient_accumulation_steps=2 --gradient_checkpointing --learning_rate=1e-4 --lr_scheduler="constant"  --lr_warmup_steps=0 --use_8bit_adam --max_train_steps=500 --checkpointing_steps=717 --seed=42
```

## Inference

In [ ]:
from diffusers import DiffusionPipeline, AutoencoderKL
import torch

var = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=var,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe.load_lora_weights("models/lora_xl_model")
_ = pipe.to("cuda")


In [ ]:
from diffusers import (
    DDPMScheduler,
    DDIMScheduler,
    PNDMScheduler,
    LMSDiscreteScheduler,
    EulerDiscreteScheduler,
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
)

repo_id = "stabilityai/stable-diffusion-xl-base-1.0"

# ddpm = DDPMScheduler.from_pretrained(repo_id, subfolder="scheduler")
# ddim = DDIMScheduler.from_pretrained(repo_id, subfolder="scheduler")
# pndm = PNDMScheduler.from_pretrained(repo_id, subfolder="scheduler")
# lms = LMSDiscreteScheduler.from_pretrained(repo_id, subfolder="scheduler")
# euler = EulerDiscreteScheduler.from_pretrained(repo_id, subfolder="scheduler")
euler_anc = EulerAncestralDiscreteScheduler.from_pretrained(repo_id, subfolder="scheduler")
# dpm = DPMSolverMultistepScheduler.from_pretrained(repo_id, subfolder="scheduler")

pipe.scheduler = euler_anc

In [ ]:
image = pipe("A portrait of sks woman as space adventurer, very detailed, ultradefined", guidance_scale=7.5, num_inference_steps=50).images[0]
image.save("images/output_lora_xl_1.png")

In [ ]:
image = pipe("A portrait of sks woman as space adventurer, very detailed, ultradefined, artstation, trending on artstation, 8k, HQ, sharp focus, ultra detailed, cinematic lighting", guidance_scale=7.5, num_inference_steps=50).images[0]
image.save("images/output_lora_xl_2.png")

In [ ]:
image = pipe("A portrait of sks woman in the style of Rembrandt, dramatic lighting").images[0]
image.save("images/output_lora_xl_3.png")

In [ ]:
image = pipe("A portrait of sks woman in ghost in the shell, detailed scene, red, perfect face, intricately detailed photorealism, trending on artstation, neon lights, rainy day", guidance_scale=7.5, num_inference_steps=50).images[0]
image.save("images/output_lora_xl_4.png")